In [ ]:
!pip install -q sacrebleu

In [ ]:
import ast, copy, gzip, json, random, subprocess, sys, tempfile, os, io, tokenize, difflib, statistics, urllib.request
from concurrent.futures import ThreadPoolExecutor
import sacrebleu

HERE = os.getcwd()
os.makedirs(HERE + "/tmp", exist_ok=True)
URL = "https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz"
raw = gzip.decompress(urllib.request.urlopen(URL).read()).decode()
problems = [json.loads(l) for l in raw.splitlines()]
print(len(problems), "bài")

164 bài


In [ ]:
def run_program(src, timeout=10):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False, dir=HERE + "/tmp") as f:
        f.write(src)
        path = f.name
    try:
        r = subprocess.run([sys.executable, "-I", path], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)



def find_entry(tree, name):
    cands = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == name]
    return cands[-1]


def strip_doc(fn):
    if fn.body and isinstance(fn.body[0], ast.Expr) and isinstance(getattr(fn.body[0], "value", None), ast.Constant) \
            and isinstance(fn.body[0].value.value, str):
        fn.body = fn.body[1:] or [ast.Pass()]

In [ ]:
def local_names(fn):
    names = set(a.arg for a in ast.walk(fn) if isinstance(a, ast.arg))
    for n in ast.walk(fn):
        if isinstance(n, ast.Name) and isinstance(n.ctx, (ast.Store, ast.Del)):
            names.add(n.id)
    return names


def rename_locals(fn):
    names = sorted(local_names(fn))
    mp = {n: f"v{i}" for i, n in enumerate(names)}

    class R(ast.NodeTransformer):
        def visit_Name(self, node):
            if node.id in mp:
                node.id = mp[node.id]
            return node

        def visit_arg(self, node):
            if node.arg in mp:
                node.arg = mp[node.arg]
            return node
    return R().visit(fn)

In [ ]:
MUT_CMP = {ast.Lt: ast.LtE, ast.LtE: ast.Lt, ast.Gt: ast.GtE, ast.GtE: ast.Gt, ast.Eq: ast.NotEq,
           ast.NotEq: ast.Eq, ast.In: ast.NotIn, ast.NotIn: ast.In}
MUT_BIN = {ast.Add: ast.Sub, ast.Sub: ast.Add, ast.Mult: ast.Add, ast.FloorDiv: ast.Mult, ast.Mod: ast.Mult}


def mutation_sites(fn):
    sites = []
    for i, n in enumerate(ast.walk(fn)):
        if isinstance(n, ast.Compare):
            for j, op in enumerate(n.ops):
                if type(op) in MUT_CMP:
                    sites.append((i, "cmp", j))
        elif isinstance(n, ast.BinOp) and type(n.op) in MUT_BIN:
            sites.append((i, "bin", 0))
        elif isinstance(n, ast.BoolOp):
            sites.append((i, "bool", 0))
        elif isinstance(n, ast.Constant) and type(n.value) is int:
            sites.append((i, "int", 0))
        elif isinstance(n, ast.Constant) and type(n.value) is bool:
            sites.append((i, "bool_const", 0))
    return sites


def apply_mutation(fn, site):
    fn = copy.deepcopy(fn)
    idx, kind, j = site
    node = list(ast.walk(fn))[idx]
    if kind == "cmp":
        node.ops[j] = MUT_CMP[type(node.ops[j])]()
    elif kind == "bin":
        node.op = MUT_BIN[type(node.op)]()
    elif kind == "bool":
        node.op = ast.Or() if isinstance(node.op, ast.And) else ast.And()
    elif kind == "int":
        node.value = node.value + 1
    elif kind == "bool_const":
        node.value = not node.value
    return fn

In [ ]:
def build_program(module, entry, new_fn):
    m = copy.deepcopy(module)
    for k, n in enumerate(m.body):
        if isinstance(n, ast.FunctionDef) and n.name == entry and n is not None:
            last = k
    m.body[last] = new_fn
    return ast.unparse(ast.fix_missing_locations(m))



def tokens(src):
    out = []
    for t in tokenize.generate_tokens(io.StringIO(src).readline):
        if t.type in (tokenize.NEWLINE, tokenize.NL, tokenize.INDENT, tokenize.DEDENT, tokenize.ENDMARKER, tokenize.COMMENT):
            continue
        out.append(t.string)
    return out


def metrics(ref_src, cand_src):
    rt, ct = tokens(ref_src), tokens(cand_src)
    bleu = sacrebleu.sentence_bleu(" ".join(ct), [" ".join(rt)], tokenize="none").score
    em = float(ref_src.strip() == cand_src.strip())
    sim = difflib.SequenceMatcher(None, rt, ct).ratio() * 100
    return bleu, em, sim

In [ ]:
tests_ok = lambda prog, p: run_program(prog + "\n\n" + p["test"] + f"\n\ncheck({p['entry_point']})\n")

rows = []
os.makedirs(HERE + "/tmp", exist_ok=True)


def process(p):
    src = p["prompt"] + p["canonical_solution"]
    module = ast.parse(src)
    entry = p["entry_point"]
    fn = find_entry(module, entry)
    ref_fn = strip_doc(copy.deepcopy(fn)) or None
    ref_fn = copy.deepcopy(fn); strip_doc(ref_fn)
    ref_src = ast.unparse(ref_fn)
    out = []
    base_prog = build_program(module, entry, ref_fn)
    if not tests_ok(base_prog, p):
        return out, "reference_fails"
    # renamed variant
    rn = rename_locals(copy.deepcopy(ref_fn))
    rn_src = ast.unparse(rn)
    if rn_src != ref_src:
        ok = tests_ok(build_program(module, entry, rn), p)
        b, e, s = metrics(ref_src, rn_src)
        out.append(dict(task=p["task_id"], kind="renamed", correct=ok, bleu=b, em=e, sim=s))
    # mutants
    sites = mutation_sites(ref_fn)
    random.Random(int(p['task_id'].split('/')[1])).shuffle(sites)
    killed = survived = 0
    for site in sites:
        if killed >= 5:
            break
        mfn = apply_mutation(ref_fn, site)
        msrc = ast.unparse(mfn)
        if msrc == ref_src:
            continue
        ok = tests_ok(build_program(module, entry, mfn), p)
        if ok:
            survived += 1
            continue
        killed += 1
        b, e, s = metrics(ref_src, msrc)
        out.append(dict(task=p["task_id"], kind="mutant", correct=False, bleu=b, em=e, sim=s))
    return out, f"survived={survived}"


with ThreadPoolExecutor(2) as ex:
    results = list(ex.map(process, problems))
skipped = [p["task_id"] for p, (o, msg) in zip(problems, results) if msg == "reference_fails"]
rows = [r for o, _ in results for r in o]

json.dump(rows, open(os.path.join(HERE, "tn1_rows.json"), "w"))
print("problems", len(problems), "skipped(reference fails after unparse):", skipped)

problems 164 skipped(reference fails after unparse): []


In [ ]:
ren = [r for r in rows if r["kind"] == "renamed"]
ren_ok = [r for r in ren if r["correct"]]
mut = [r for r in rows if r["kind"] == "mutant"]
print("renamed variants:", len(ren), "of which pass tests:", len(ren_ok))
print("killed mutants:", len(mut), "from", len({r['task'] for r in mut}), "problems")


def summ(name, xs):
    for key in ("bleu", "sim"):
        v = [x[key] for x in xs]
        print(f"{name:28s} {key:5s} mean={statistics.mean(v):6.1f} median={statistics.median(v):6.1f} min={min(v):6.1f} max={max(v):6.1f}")
    print(f"{name:28s} EM    mean={statistics.mean(x['em'] for x in xs):.3f}")


summ("correct (renamed)", ren_ok)
summ("incorrect (killed mutant)", mut)

# AUC: probability that a random correct candidate scores higher than a random incorrect one
def auc(pos, neg):
    wins = sum((p > n) + 0.5 * (p == n) for p in pos for n in neg)
    return wins / (len(pos) * len(neg))

for key in ("bleu", "sim", "em"):
    print(f"AUC({key}) correct-vs-incorrect = {auc([x[key] for x in ren_ok], [x[key] for x in mut]):.3f}")

# Paired comparison per problem: renamed BLEU < mutant BLEU ?
by = {}
for r in rows:
    by.setdefault(r["task"], {}).setdefault(r["kind"], []).append(r)
pairs = [(statistics.mean(x["bleu"] for x in d["renamed"]), statistics.mean(x["bleu"] for x in d["mutant"]))
         for d in by.values() if "renamed" in d and "mutant" in d and d["renamed"][0]["correct"]]
worse = sum(a < b for a, b in pairs)
print(f"problems with both: {len(pairs)}; BLEU(correct renamed) < mean BLEU(buggy mutants) in {worse} ({100*worse/len(pairs):.1f}%)")

for thr in (80, 90):
    fn_hi = sum(x["bleu"] >= thr for x in mut)
    fp_lo = sum(x["bleu"] < thr for x in ren_ok)
    print(f"threshold BLEU>={thr}: buggy accepted = {fn_hi}/{len(mut)} ({100*fn_hi/len(mut):.1f}%), correct rejected = {fp_lo}/{len(ren_ok)} ({100*fp_lo/len(ren_ok):.1f}%)")
print("EM of correct renamed = 0 for", sum(x["em"] == 0 for x in ren_ok), "of", len(ren_ok))

renamed variants: 164 of which pass tests: 164
killed mutants: 590 from 150 problems
correct (renamed)            bleu  mean=  46.3 median=  45.2 min=   7.1 max=  92.8
correct (renamed)            sim   mean=  78.5 median=  78.5 min=  54.3 max=  90.9
correct (renamed)            EM    mean=0.000
incorrect (killed mutant)    bleu  mean=  94.7 median=  95.5 min=  57.9 max=  98.9
incorrect (killed mutant)    sim   mean=  98.0 median=  98.3 min=  86.7 max=  99.5
incorrect (killed mutant)    EM    mean=0.000
AUC(bleu) correct-vs-incorrect = 0.001
AUC(sim) correct-vs-incorrect = 0.000
AUC(em) correct-vs-incorrect = 0.500
problems with both: 150; BLEU(correct renamed) < mean BLEU(buggy mutants) in 150 (100.0%)
threshold BLEU>=80: buggy accepted = 587/590 (99.5%), correct rejected = 163/164 (99.4%)
threshold BLEU>=90: buggy accepted = 556/590 (94.2%), correct rejected = 163/164 (99.4%)
EM of correct renamed = 0 for 164 of 164
